<a href="https://colab.research.google.com/github/Nazmussakibnipun/ml_final_project/blob/main/finalml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Data Loading (5 Marks)

In [26]:
import pandas as pd

df = pd.read_csv('/content/vgsales.csv')

print(df.head())

print(df.shape)

   Rank                      Name Platform    Year         Genre Publisher  \
0     1                Wii Sports      Wii  2006.0        Sports  Nintendo   
1     2         Super Mario Bros.      NES  1985.0      Platform  Nintendo   
2     3            Mario Kart Wii      Wii  2008.0        Racing  Nintendo   
3     4         Wii Sports Resort      Wii  2009.0        Sports  Nintendo   
4     5  Pokemon Red/Pokemon Blue       GB  1996.0  Role-Playing  Nintendo   

   NA_Sales  EU_Sales  JP_Sales  Other_Sales  Global_Sales  
0     41.49     29.02      3.77         8.46         82.74  
1     29.08      3.58      6.81         0.77         40.24  
2     15.85     12.88      3.79         3.31         35.82  
3     15.75     11.01      3.28         2.96         33.00  
4     11.27      8.89     10.22         1.00         31.37  
(16598, 11)


# 2. Data Preprocessing (10 Marks)

In [27]:
from sklearn.preprocessing import StandardScaler

df = df.dropna()

df = df[['NA_Sales','EU_Sales','JP_Sales','Other_Sales','Global_Sales']]

df['Total_Sales'] = df['NA_Sales'] + df['EU_Sales'] + df['JP_Sales'] + df['Other_Sales']

df = df[df['Total_Sales'] < df['Total_Sales'].quantile(0.99)]

scaler = StandardScaler()
X = df.drop('Global_Sales', axis=1)
y = df['Global_Sales']

X = scaler.fit_transform(X)

# 3. Pipeline Creation (10 Marks)

In [28]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

pipeline = Pipeline([
    ('model', RandomForestRegressor(random_state=42))
])

# **4. Primary Model Selection (5 Marks)**

Random Forest is chosen because it handles non-linear relationships, works well with mixed feature types, and is robust to overfitting.

# 5. Model Training (10 Marks)

In [29]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

pipeline.fit(X_train, y_train)

Pipeline(steps=[('model', RandomForestRegressor(random_state=42))])

# 6. Cross-Validation (10 Marks)

In [30]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(pipeline, X_train, y_train, cv=5)

print("Mean CV Score:", scores.mean())
print("Std Dev:", scores.std())

Mean CV Score: 0.9999327675936266
Std Dev: 2.7765223453586753e-06


# 7. Hyperparameter Tuning (10 Marks)

In [31]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'model__n_estimators': [50, 100],
    'model__max_depth': [None, 10],
    'model__min_samples_split': [2, 5]
}

grid = GridSearchCV(pipeline, param_grid, cv=3, scoring='r2')
grid.fit(X_train, y_train)

print("Tested Params:", param_grid)
print("Best Params:", grid.best_params_)
print("Best Score:", grid.best_score_)

Tested Params: {'model__n_estimators': [50, 100], 'model__max_depth': [None, 10], 'model__min_samples_split': [2, 5]}
Best Params: {'model__max_depth': 10, 'model__min_samples_split': 2, 'model__n_estimators': 50}
Best Score: 0.999928717549031


# 8. Best Model Selection (10 Marks)

In [32]:
best_model = grid.best_estimator_

# 9. Model Performance Evaluation (10 Marks)

In [33]:
from sklearn.metrics import mean_squared_error, r2_score

y_pred = best_model.predict(X_test)

print("MSE:", mean_squared_error(y_test, y_pred))
print("R2 Score:", r2_score(y_test, y_pred))

MSE: 3.0242869693193607e-05
R2 Score: 0.9999434295910777


# 10. Web Interface with Gradio (10 Marks)

In [34]:
import gradio as gr
import numpy as np

def predict_sales(na, eu, jp, other):
    total = na + eu + jp + other

    input_data = np.array([[na, eu, jp, other, total]])
    input_data = scaler.transform(input_data)

    return best_model.predict(input_data)[0]

interface = gr.Interface(
    fn=predict_sales,
    inputs=[
        gr.Number(label="NA Sales"),
        gr.Number(label="EU Sales"),
        gr.Number(label="JP Sales"),
        gr.Number(label="Other Sales")
    ],
    outputs="number",
    title="Global Video Game Sales Predictor"
)

interface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d9f6236ac3113e914c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# 11. Deployment to Hugging Face (10 Marks)